# Arkansas — Title 23, Subtitle 3 (Insurance) → `data/arkansas/ins_codes/*.md`

Arkansas insurance statutes are **Subtitle 3 — INSURANCE** under **Title 23** of the *Code of Arkansas* (1987), sections roughly **§§ 23-60-101 — 23-103-417**.

The legislature’s public site routes the codified statutes through **Lexis** (`advance.lexis.com`), which is not practical to scrape with plain HTTP. This notebook instead pulls **HTML** from **Justia’s** mirror of the Arkansas Code ([Title 23, Subtitle 3](https://law.justia.com/codes/arkansas/title-23/subtitle-3/)). Justia is protected by **Cloudflare**; unauthenticated **`httpx`** is usually blocked. We use **`curl_cffi`** with a browser TLS fingerprint (`impersonate="chrome120"`) so requests succeed the same way a normal browser would.

**Authoritative / official:** verify current text with the state’s **Lexis** public access portal ([Arkansas Code Search — portal.arkansas.gov](https://portal.arkansas.gov/service/arkansas-code-search-laws-and-statutes/)) or the General Assembly’s [Arkansas Law](https://www.arkleg.state.ar.us/ArkansasLaw) hub.

This notebook:

1. Crawls the Subtitle 3 index → **chapter** pages → **subchapter** pages (where present) and collects every **`section-23-…`** URL.
2. Downloads each section page, extracts text from **`div.primary-content`**, and writes **`ACA_sec_<section>.md`** under **`data/arkansas/ins_codes/`** (A.C.A. = Arkansas Code Annotated).

**Volume:** about **2,000** sections — a full run can take **~15–25 minutes** with the default delay. Use **`MAX_SECTIONS`** to cap **downloads** while testing; **discovery** still walks every chapter/subchapter page unless you reuse a saved URL list (see **`REUSE_DISCOVERED_URLS`**).

**Politeness:** **`REQUEST_DELAY_SEC`** (default **0.12 s**) between HTTP requests during discovery and download.

Then run **`python -m app.ingest`** from the project root.

In [1]:
%pip install -q curl_cffi beautifulsoup4

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
SUBTITLE_INDEX = f"{BASE}/codes/arkansas/title-23/subtitle-3/"

OUT_DIR = Path("data") / "arkansas" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Browser TLS impersonation (Cloudflare on Justia blocks many plain-HTTP clients).
CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

# 0 = all discovered sections; set e.g. 40 to smoke-test.
MAX_SECTIONS = 0

SKIP_EXISTING = True

# If True and `_arkansas_subtitle3_section_urls.txt` already exists in OUT_DIR, skip the crawl and load URLs from that file (delete the file to force a fresh discovery).
REUSE_DISCOVERED_URLS = True

chapter_re = re.compile(r"^/codes/arkansas/title-23/subtitle-3/chapter-\d+/$")
subch_re = re.compile(r"^/codes/arkansas/title-23/subtitle-3/chapter-\d+/subchapter-\d+/$")
sec_re = re.compile(
    r"^/codes/arkansas/title-23/subtitle-3/chapter-\d+(?:/subchapter-\d+)?/section-23-\d+-\d+/$"
)
section_id_re = re.compile(r"/section-(23-\d+-\d+)/?$", re.I)

In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def discover_section_urls() -> list[str]:
    """BFS chapter + subchapter pages; return sorted unique absolute section URLs."""
    html = curl_get(SUBTITLE_INDEX)
    soup = BeautifulSoup(html, "html.parser")
    queue: list[str] = []
    for a in soup.find_all("a", href=True):
        h = a["href"]
        if chapter_re.match(h):
            queue.append(urljoin(BASE, h))

    seen_pages: set[str] = set()
    sections: set[str] = set()
    while queue:
        page_url = queue.pop(0)
        if page_url in seen_pages:
            continue
        seen_pages.add(page_url)
        body = curl_get(page_url)
        sp = BeautifulSoup(body, "html.parser")
        for a in sp.find_all("a", href=True):
            h = a["href"]
            if subch_re.match(h):
                queue.append(urljoin(BASE, h))
            elif sec_re.match(h):
                sections.add(urljoin(BASE, h))

    return sorted(sections)


def section_label_from_url(url: str) -> str:
    m = section_id_re.search(urlparse(url).path)
    if not m:
        raise ValueError(f"cannot parse section id from {url!r}")
    return m.group(1)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"ACA_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    """Return (page <title>, plain text from div.primary-content or fallback)."""
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Arkansas Code" in s and "(" in s:
            # year picker lines like "2024 Arkansas Code (here)"
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("AR Code §"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_subtitle3() -> dict[str, int]:
    list_path = OUT_DIR / "_arkansas_subtitle3_section_urls.txt"
    if REUSE_DISCOVERED_URLS and list_path.exists() and list_path.stat().st_size > 50:
        all_urls = [ln.strip() for ln in list_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
        print(f"Loaded {len(all_urls)} section URLs from {list_path.name} (skipped discovery)")
    else:
        all_urls = discover_section_urls()
        print(f"Discovered {len(all_urls)} section URLs under {SUBTITLE_INDEX}")
        list_path.write_text("\n".join(all_urls), encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote, skipped, failed = 0, 0, 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Arkansas Code § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Arkansas Code — Title 23, Subtitle 3 (Insurance)**\n\n"
                    f"**Source (mirror):** {sec_url}\n\n"
                    f"**Section:** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_subtitle3()

Discovered 1964 section URLs under https://law.justia.com/codes/arkansas/title-23/subtitle-3/
… 200/1964 (wrote=200 skipped=0 failed=0)
… 400/1964 (wrote=400 skipped=0 failed=0)
… 600/1964 (wrote=600 skipped=0 failed=0)
… 800/1964 (wrote=800 skipped=0 failed=0)
FAIL 23-74-303: Failed to perform, curl: (28) Operation timed out after 60002 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
… 1000/1964 (wrote=999 skipped=0 failed=1)
… 1200/1964 (wrote=1199 skipped=0 failed=1)
… 1400/1964 (wrote=1399 skipped=0 failed=1)
… 1600/1964 (wrote=1599 skipped=0 failed=1)
… 1800/1964 (wrote=1799 skipped=0 failed=1)
Done. wrote=1963 skipped=0 failed=1 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/arkansas/ins_codes


{'wrote': 1963, 'skipped': 0, 'failed': 1}

## Next step

`python -m app.ingest` from the repository root.